# AegisArt (APF3 · Ruta A) — Investigación, Entrenamiento y Validación del Modelo de IA para Moderación y Detección de Plagio (US-09)

**Proyecto:** AegisArt — Plataforma de protección de obras digitales ante IA generativa no autorizada y monitoreo de plagio.  
**Curso:** Innovación y Transformación Digital (Ciclo VIII — UTP).  
**Enfoque:** **Ruta A (Implementación de IA Real)** integrada en el backend FastAPI y visible en el frontend Angular.

---

## 1. Definición del Problema y Justificación ML (Supervisado vs. No Supervisado)

En AegisArt, cuando un usuario sube una obra nueva (`POST /api/obras/proteger-y-registrar`), la Historia de Usuario **US-09** requiere decidir automáticamente si la obra debe aprobarse o enviarse a la cola de moderación (**US-10**) por riesgo de:
1. **Copia o plagio visual/semántico no declarado** respecto al catálogo existente.
2. **Evasión fraudulenta mediante marca de agua ajena** (`AEGIS-DCT-*` extraída de la imagen subida).
3. **Contenido anómalo, corrupto o spam de baja/extrema entropía**.

Al mismo tiempo, **US-08** permite que un artista publique legítimamente una **obra derivada declarada** (`obra_inspirada_id > 0`), la cual presenta similitud alta (`65%–92%`) con la obra original pero **NO** es plagio si no porta una marca invisible ajena ni supera un clonado directo.

### Hipótesis experimental evaluada
En vez de asumir de antemano si conviene **Aprendizaje No Supervisado** o **Aprendizaje Supervisado**, evaluamos objetivamente ambos paradigmas sobre el mismo conjunto de datos real:
- **6 clasificadores supervisados:** Árbol de Decisión (DT), Random Forest, **Gradient Boosting**, Regresión Logística, KNN (k=5) y SVM (RBF).
- **2 algoritmos no supervisados:** K-Means (k=2) e Isolation Forest.

In [1]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
from PIL import Image as PILImage

try:
    from IPython.display import Image as IPyImage, display
    TIENE_IPYTHON = True
except ImportError:
    TIENE_IPYTHON = False

SERVIDOR_DIR = Path.cwd().resolve()
if SERVIDOR_DIR.name == "ia_experimentos":
    SERVIDOR_DIR = SERVIDOR_DIR.parent
if str(SERVIDOR_DIR) not in sys.path:
    sys.path.insert(0, str(SERVIDOR_DIR))

from ia_experimentos.entrenar_y_comparar_apf3 import (
    FEATURES_MODELO,
    RUTA_DATASET_CSV,
    RUTA_METRICAS_JSON,
    RUTA_MODELO_PKL,
    CARPETA_GRAFICAS,
    construir_corpus_y_dataset,
    ejecutar_benchmark_desde_dataframe,
    generar_obra_artistica_base,
)
from aplicacion.servicios.servicio_ia_apf3 import predecir_obra_sospechosa_ml

print(f"Directorio backend: {SERVIDOR_DIR}")
print(f"Dataset CSV existe: {RUTA_DATASET_CSV.exists()} -> {RUTA_DATASET_CSV}")
print(f"Modelo PKL existe:  {RUTA_MODELO_PKL.exists()} -> {RUTA_MODELO_PKL}")

Directorio backend: D:\UTP\Ciclo VIII\Innovación y transformación digital\Proyecto\APF3\proyecto\aegisart\servidor_backend
Dataset CSV existe: True -> D:\UTP\Ciclo VIII\Innovación y transformación digital\Proyecto\APF3\proyecto\aegisart\servidor_backend\aplicacion\nucleo\dataset_obras_aegisart.csv
Modelo PKL existe:  True -> D:\UTP\Ciclo VIII\Innovación y transformación digital\Proyecto\APF3\proyecto\aegisart\servidor_backend\aplicacion\nucleo\modelo_ia_apf3.pkl


## 2. Carga del Corpus Real, Limpieza de Datos y Análisis Exploratorio (EDA)

El dataset (`dataset_obras_aegisart.csv`) proviene de **300 muestras reales** derivadas de **30 obras artísticas base distintas** (paisajes pastel, retratos cálidos, líneas de tinta, mosaicos op-art, constelaciones, ondas de terreno y composiciones surrealistas), sometidas a transformaciones reales (compresión JPEG, recorte 20%, desenfoque + brillo, reinserción tras marca de agua `AEGIS-DCT` DWT/DCT y anomalías visuales), con extracción de **512 dimensiones CLIP (`openai/clip-vit-base-patch32`)**, **pHash perceptual**, **esteganografía DCT/DWT**, y **features OpenCV**.

Se verifica:
- Ausencia de valores nulos (`NaN`) y duplicados exactos.
- Balance perfecto de clases (`150` obras seguras vs. `150` sospechosas).
- Estadísticas descriptivas reales por clase objetivo.

In [2]:
if RUTA_DATASET_CSV.exists():
    df = pd.read_csv(RUTA_DATASET_CSV)
else:
    df = construir_corpus_y_dataset()
    df.to_csv(RUTA_DATASET_CSV, index=False, encoding="utf-8")

print("Dimensiones del dataset:", df.shape)
print("Obras base únicas:", df["grupo_obra_base"].nunique())
print("Valores nulos totales:", int(df.isna().sum().sum()))
print("Duplicados exactos:", int(df.duplicated(subset=FEATURES_MODELO + ["es_obra_sospechosa_target"]).sum()))
print("\nDistribución de clases (0=Segura, 1=Sospechosa):")
print(df["es_obra_sospechosa_target"].value_counts().to_string())
print("\nPromedios reales por clase objetivo:")
print(df.groupby("es_obra_sospechosa_target")[FEATURES_MODELO].mean().round(4).T.to_string())

Dimensiones del dataset: (300, 13)
Obras base únicas: 30
Valores nulos totales: 0
Duplicados exactos: 0

Distribución de clases (0=Segura, 1=Sospechosa):
es_obra_sospechosa_target
0    150
1    150

Promedios reales por clase objetivo:
es_obra_sospechosa_target            0        1
similitud_phash                72.9747  87.0320
similitud_clip                 91.3020  91.7127
similitud_maxima_catalogo      81.2225  89.1397
declarada_inspirada_us08        0.4000   0.1000
marca_ajena_detectada           0.0000   0.2000
densidad_bordes_canny           0.0721   0.0821
contraste_luminancia            0.2912   0.2509
entropia_visual                 5.5463   5.1501
proporcion_tonos_calidos_piel   0.2369   0.3085
saturacion_media_hsv            0.3451   0.3571


## 3. Partición Group-Aware (80% / 20%) y Comparativa Supervisado vs. No Supervisado

Para evitar **fuga de datos (*data leakage*)** entre variantes de una misma obra, la partición 80/20 se realiza con `GroupShuffleSplit` agrupado por `grupo_obra_base` (`240` muestras / `24` obras base para entrenamiento y `60` muestras / `6` obras base completamente no vistas para prueba), además de **validación cruzada estratificada de 5 pliegues (5-Fold CV)** y **11 pruebas de cordura fuera de muestra (OOD)** con estilos difíciles (tinta densa, pastel bajo contraste, acuarela, retrato cálido, copia con `AEGIS-DCT`, etc.).

In [3]:
metricas = ejecutar_benchmark_desde_dataframe(df, generar_graficas=True)
df_comp = pd.DataFrame(metricas["comparativa_algoritmos"])
print("\n=== Tabla comparativa final (Supervisado vs. No Supervisado) ===")
print(df_comp.to_string(index=False))
print("\nModelo ganador en producción:", metricas["algoritmo"])
print("Matriz de confusión (Test 20% = 60 muestras):", metricas["matriz_confusion"])
print("ROC-AUC Test:", metricas["roc_auc_porcentaje"], "% | CV F1 (5-Fold):", metricas["cv_f1_entrenamiento_porcentaje"], "% | Cordura OOD:", metricas["cordura_dominio"])


4. Benchmark de modelos (Train=240 / Test=60 obras base no vistas):
  - Árbol de Decisión (DT)     | Acc=86.67% | Prec=86.67% | Rec=86.67% | F1=86.67% | CV_F1=86.17%
  - Random Forest              | Acc=95.00% | Prec=100.00% | Rec=90.00% | F1=94.74% | CV_F1=91.51%
  - Gradient Boosting          | Acc=93.33% | Prec=93.33% | Rec=93.33% | F1=93.33% | CV_F1=97.46%
  - Regresión Logística        | Acc=96.67% | Prec=96.67% | Rec=96.67% | F1=96.67% | CV_F1=93.70%
  - K-Vecinos (KNN, k=5)       | Acc=93.33% | Prec=100.00% | Rec=86.67% | F1=92.86% | CV_F1=95.55%
  - SVM (Kernel RBF)           | Acc=96.67% | Prec=100.00% | Rec=93.33% | F1=96.55% | CV_F1=94.43%
  - K-Means (k=2)              | Acc=60.00% | Prec=57.14% | Rec=80.00% | F1=66.67% | Sil=0.3736
  - Isolation Forest           | Acc=65.00% | Prec=60.00% | Rec=90.00% | F1=72.00% | Sil=None

4B. Pruebas de cordura en casos reales fuera de muestra (11 escenarios críticos):
  - Árbol de Decisión (DT)     | Cordura=7/11 | Fallos=['Arte linea

## 4. Evidencia Visual: Gráficas Comparativas, PCA y Curva de Aprendizaje

A continuación se muestran las 5 gráficas generadas a partir de los resultados experimentales reales:
1. Comparativa de F1-Score y Accuracy (6 supervisados vs. 2 no supervisados).
2. Matriz de confusión del modelo ganador (**Gradient Boosting**).
3. Importancia de las 10 características extraídas.
4. Proyección PCA 2D comparando las etiquetas reales vs. agrupamiento no supervisado **K-Means (k=2)**, donde se observa por qué K-Means agrupa erróneamente derivadas legítimas US-08 junto con plagios.
5. Curva de aprendizaje (Train CV vs. Validación CV) verificando ausencia de sobreajuste.

In [4]:
rutas_png = [
    "comparativa_modelos_ia.png",
    "matriz_confusion_apf3.png",
    "importancia_caracteristicas.png",
    "clustering_vs_supervisado_pca.png",
    "curva_aprendizaje_apf3.png",
]
for nombre in rutas_png:
    ruta = CARPETA_GRAFICAS / nombre
    with PILImage.open(ruta) as im:
        w, h = im.size
    print(f"Gráfica verificada: {nombre} ({w}x{h}px, {ruta.stat().st_size} bytes)")
    if TIENE_IPYTHON:
        display(IPyImage(filename=str(ruta)))


Gráfica verificada: comparativa_modelos_ia.png (1120x620px, 34826 bytes)
Gráfica verificada: matriz_confusion_apf3.png (760x560px, 30580 bytes)
Gráfica verificada: importancia_caracteristicas.png (900x540px, 30919 bytes)
Gráfica verificada: clustering_vs_supervisado_pca.png (960x500px, 21989 bytes)
Gráfica verificada: curva_aprendizaje_apf3.png (900x520px, 19861 bytes)


## 5. Prueba de Inferencia en Producción y Limitaciones Honestamente Documentadas

Se prueba directamente la función de producción `predecir_obra_sospechosa_ml(...)` usada por FastAPI en `/api/obras/proteger-y-registrar` sobre tres escenarios reales:
- **Escenario A:** Obra original nueva.
- **Escenario B:** Obra derivada declarada legítimamente bajo **US-08** (`similitud_clip=88%`, `declarada_inspirada=True`, sin marca ajena).
- **Escenario C:** Plagio visual/semántico sin declarar (`similitud_clip=95%`, `declarada_inspirada=False`).

### Limitaciones y riesgos conocidos
- En grabados con densidad extrema de bordes (`densidad_bordes_canny >= 0.30`), el clasificador puede elevar falso positivo y requerir revisión humana (**US-10**).
- Conforme crezca la base real de moderación en producción, el modelo `modelo_ia_apf3.pkl` debe reentrenarse periódicamente incorporando las resoluciones confirmadas por los moderadores en PostgreSQL.

In [5]:
import tempfile

with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
    ruta_prueba = tmp.name

try:
    generar_obra_artistica_base(seed=25, tema="pastel_minimalista").save(ruta_prueba)

    escenario_a = predecir_obra_sospechosa_ml(
        ruta_imagen=ruta_prueba,
        titulo="Paisaje pastel original",
        similitud_maxima=25.0,
        declarada_inspirada=False,
        similitud_phash=22.0,
        similitud_clip=28.0,
        marca_ajena_detectada=False,
    )
    escenario_b = predecir_obra_sospechosa_ml(
        ruta_imagen=ruta_prueba,
        titulo="Homenaje declarado US-08",
        similitud_maxima=86.0,
        declarada_inspirada=True,
        similitud_phash=84.0,
        similitud_clip=88.0,
        marca_ajena_detectada=False,
    )
    escenario_c = predecir_obra_sospechosa_ml(
        ruta_imagen=ruta_prueba,
        titulo="Copia sin declaracion",
        similitud_maxima=94.0,
        declarada_inspirada=False,
        similitud_phash=92.0,
        similitud_clip=95.0,
        marca_ajena_detectada=False,
    )
    for etiqueta, res in [("Escenario A (Original)", escenario_a), ("Escenario B (US-08 Derivada)", escenario_b), ("Escenario C (Plagio sin US-08)", escenario_c)]:
        print(f"{etiqueta} -> sospechosa={res['es_sospechosa']} | {res['evaluacion_resumen']}")
finally:
    if os.path.exists(ruta_prueba):
        os.remove(ruta_prueba)

Escenario A (Original) -> sospechosa=False | Obra legítima aprobada · 98.8% confianza ML
Escenario B (US-08 Derivada) -> sospechosa=False | Obra legítima aprobada · 99.5% confianza ML
Escenario C (Plagio sin US-08) -> sospechosa=True | Sospechosa · 89.4% riesgo ML
